# Jointly Optimized PPO Portfolio Management with a Differentiable Regime Encoder
### Colombo Stock Exchange (CSE) — Banking Sector, 2021–2025

End-to-end pipeline notebook. Runs top-to-bottom on **Colab or locally**.

---

## Architecture

```
Market Observation (Prices, Volumes, Macro)
        │
        ▼
Preprocessing Module  (feature engineering → normalization → 30-day windowing)
        │   x_t : (30, n_features)
        ▼
┌──────────────── Differentiable Regime Encoder ─────────────────┐
│   V-VSN  ──►  LSTM  ──►  Temporal Attention (MHA)              │
│                                    │                            │
│                     Macro Graph Prior (GNN/GAT)                 │
│                                    │                            │
│                         Fusion ────┴──► h_temp  +  regime_probs │
└─────────────────────────────────────────────────────────────────┘
        │                                    │
        ▼                                    ▼
  Actor (π)  ──► a_t weights          Critic (V) ──► V(s)
        │                                    │
        ▼                                    │
Market Environment ──► realized returns      │
        │                                    │
        ▼                                    ▼
Reward Module (Sharpe − Costs − EVaR) ──► GAE ──► PPO clipped loss
        │
        └──────────► joint end-to-end backward through EVERYTHING
```

**Note on the MHA:** the attention block consumes the **LSTM output sequence**, not
the raw variable-selection output. Data flow is strictly `VSN → LSTM → MHA`.

---

## Contents

| § | Section |
|---|---------|
| 1 | Environment setup & dependencies |
| 2 | Imports, seeding, device |
| 3 | Configuration |
| 4 | Data loading (CSE banking sector) |
| 5 | Feature engineering |
| 6 | Chronological splits, normalization, windowing |
| 7 | Market environments |
| 8 | Correctness patches (opt-in) |
| 9 | Model construction & shape checks |
| 10 | Training |
| 11 | Evaluation & backtest |
| 12 | Benchmarks |
| 13 | Visualization |
| 14 | Saving artifacts |

---
## 1. Environment setup & dependencies

Detects Colab vs. local. On Colab, set `REPO_URL` (or upload/mount the project) so
that `src/` and `data/raw/` are reachable.

`torch-geometric` is **not** required — the GAT in
`src/models/macro_graph_prior.py` is implemented from scratch in plain PyTorch.

In [ ]:
import sys, os, subprocess

IN_COLAB = "google.colab" in sys.modules

REQUIREMENTS = [
    "torch>=2.0.0", "numpy>=1.24.0", "pandas>=2.0.0", "scipy>=1.10.0",
    "scikit-learn>=1.2.0", "matplotlib>=3.7.0", "seaborn>=0.12.0",
    "tensorboard>=2.13.0", "pyarrow>=12.0.0", "tqdm>=4.65.0",
    "statsmodels>=0.14.0", "openpyxl>=3.1.0", "pyyaml>=6.0",
]

INSTALL_DEPS = IN_COLAB   # flip to True to force a local install

if INSTALL_DEPS:
    print("Installing dependencies ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *REQUIREMENTS],
        check=True,
    )
    print("Done.")
else:
    print("Skipping install (set INSTALL_DEPS=True to force).")

print("Colab:", IN_COLAB)

### 1.1 Locate the project root

Adjust `PROJECT_ROOT` if your layout differs. The notebook expects:

```
<PROJECT_ROOT>/
├── src/                              # the package
├── data/raw/banking_sector_2021_2025.csv
├── scripts/merge_raw_data.py
└── main.py
```

In [ ]:
from pathlib import Path

if IN_COLAB:
    # ── Option A: clone from git ────────────────────────────────────────
    REPO_URL = ""          # e.g. "https://github.com/<user>/<repo>.git"
    if REPO_URL:
        if not Path("/content/project").exists():
            subprocess.run(["git", "clone", REPO_URL, "/content/project"], check=True)
        PROJECT_ROOT = Path("/content/project")
    else:
        # ── Option B: Google Drive ──────────────────────────────────────
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        PROJECT_ROOT = Path("/content/drive/MyDrive/Regime detection portfolio")
else:
    # Local: notebook lives in <root>/notebooks/, so go one level up.
    here = Path.cwd()
    PROJECT_ROOT = here.parent if (here.parent / "src").is_dir() else here

PROJECT_ROOT = PROJECT_ROOT.resolve()
assert (PROJECT_ROOT / "src").is_dir(), f"src/ not found under {PROJECT_ROOT}"

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT :", PROJECT_ROOT)
print("Contents     :", sorted(p.name for p in PROJECT_ROOT.iterdir() if not p.name.startswith(".")))

---
## 2. Imports, seeding, device

In [ ]:
import os
import sys
import json
import logging
import warnings
import subprocess
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ── Project modules ────────────────────────────────────────────────────
from src.config.config import (
    get_config, DATA_RAW_DIR, DATA_PROCESSED_DIR,
    MODELS_DIR, RESULTS_DIR, FIGURES_DIR, LOGS_DIR,
)
from src.data.data_loader import CSEDataLoader
from src.data.preprocessor import Preprocessor
from src.environment.market_env import MarketEnvironment
from src.environment.reward import RewardCalculator
from src.models.ppo_agent import PPOAgent
from src.models.actor import ActorNetwork
from src.training.trainer import PPOTrainer
from src.utils.metrics import PortfolioMetrics

# ── Logging ────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
logging.getLogger("matplotlib").setLevel(logging.WARNING)
log = logging.getLogger("notebook")

# ── Plot style ─────────────────────────────────────────────────────────
%matplotlib inline
plt.rcParams.update({
    "figure.figsize": (13, 5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
})
sns.set_palette("husl")

# ── Reproducibility ────────────────────────────────────────────────────
SEED = 42

def set_seed(seed: int = SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch  : {torch.__version__}")
print(f"device : {DEVICE}")
if DEVICE == "cuda":
    print(f"gpu    : {torch.cuda.get_device_name(0)}")

---
## 3. Configuration

All hyperparameters come from `src/config/config.py`. Overrides for this run are
applied below — keep experiment-specific tweaks here rather than editing the
config module, so the notebook stays a reproducible record of one run.

**Macro variables are disabled** (`n_macro = 0`): the graph prior therefore builds
its adjacency over assets only.

In [ ]:
# ── Run-level knobs ────────────────────────────────────────────────────
N_ASSETS        = 47        # upper bound; the loader keeps the most complete tickers.
                            # The banking-sector file currently holds ~10 companies,
                            # so the effective count is printed in §4.
TOTAL_TIMESTEPS = 20_000    # bump to 200_000+ for a real run
ROLLOUT_LENGTH  = 256
N_EPOCHS        = 10
BATCH_SIZE      = 64
LEARNING_RATE   = 3e-4
EVAL_FREQUENCY  = 2_048
SAVE_FREQUENCY  = 5_120

config = get_config(seed=SEED, device=DEVICE)

config.data.n_assets           = N_ASSETS
config.data.n_macro_variables  = 0
config.ppo.total_timesteps     = TOTAL_TIMESTEPS
config.ppo.rollout_length      = ROLLOUT_LENGTH
config.ppo.n_epochs            = N_EPOCHS
config.ppo.batch_size          = BATCH_SIZE
config.ppo.learning_rate       = LEARNING_RATE
config.ppo.eval_frequency      = EVAL_FREQUENCY
config.ppo.save_frequency      = SAVE_FREQUENCY

# ── Summary table ──────────────────────────────────────────────────────
summary = {
    "Data": {
        "window_size": config.data.window_size,
        "train/val/test": f"{config.data.train_ratio}/{config.data.val_ratio}/{config.data.test_ratio}",
        "normalize": config.data.normalize_method,
        "n_assets": config.data.n_assets,
        "n_macro": config.data.n_macro_variables,
    },
    "Regime Encoder": {
        "vsn_hidden": config.regime_encoder.vsn_hidden_dim,
        "lstm_hidden": config.regime_encoder.lstm_hidden_dim,
        "lstm_layers": config.regime_encoder.lstm_num_layers,
        "mha_heads": config.regime_encoder.mha_num_heads,
        "gnn_hidden": config.regime_encoder.gnn_hidden_dim,
        "gnn_heads": config.regime_encoder.gnn_num_heads,
        "n_regimes": config.regime_encoder.n_regimes,
        "latent_dim": config.regime_encoder.latent_state_dim,
    },
    "PPO": {
        "clip_eps": config.ppo.clip_epsilon,
        "gamma": config.ppo.gamma,
        "gae_lambda": config.ppo.gae_lambda,
        "entropy_coeff": config.ppo.entropy_coeff,
        "value_coeff": config.ppo.value_loss_coeff,
        "lr": config.ppo.learning_rate,
        "total_timesteps": config.ppo.total_timesteps,
    },
    "Reward": {
        "risk_free": config.reward.risk_free_rate,
        "tc_rate": config.reward.transaction_cost_rate,
        "sharpe_window": config.reward.sharpe_window,
        "evar_conf": config.reward.evar_confidence,
        "evar_weight": config.reward.evar_penalty_weight,
    },
    "Environment": {
        "initial_value": config.environment.initial_portfolio_value,
        "allow_short": config.environment.allow_short_selling,
        "max_position": config.environment.max_position_size,
        "slippage": config.environment.slippage,
    },
}

for section, params in summary.items():
    print(f"\n── {section} " + "─" * (58 - len(section)))
    for k, v in params.items():
        print(f"   {k:<18} {v}")

---
## 4. Data loading

`CSEDataLoader` reads the merged raw CSV, detects the header row, pivots to
`(dates × companies)` price and volume matrices, selects the `n_assets` most
complete tickers, and forward/back-fills gaps. Results are cached to
`data/processed/*.parquet`, so re-runs are fast.

In [ ]:
merged_csv = Path(DATA_RAW_DIR) / "banking_sector_2021_2025.csv"

if not merged_csv.exists():
    print("Merged CSV missing — building it from the yearly files ...")
    subprocess.run([sys.executable, "scripts/merge_raw_data.py"], check=True, cwd=PROJECT_ROOT)

loader = CSEDataLoader(raw_data_dir=DATA_RAW_DIR)
prices, volumes = loader.load_prices_and_volumes(n_assets=config.data.n_assets)

requested = config.data.n_assets
config.data.asset_names = list(prices.columns)
config.data.n_assets    = len(config.data.asset_names)

print(f"\nPrices  : {prices.shape}   {prices.index.min().date()} → {prices.index.max().date()}")
print(f"Volumes : {volumes.shape}")
print(f"Assets  : {config.data.n_assets}  {config.data.asset_names}")
print(f"Missing : prices {prices.isna().sum().sum()}, volumes {volumes.isna().sum().sum()}")

if config.data.n_assets < requested:
    print(f"\nNOTE: requested up to {requested} assets but the raw file only contains "
          f"{config.data.n_assets}. All of them were kept; n_assets has been "
          f"updated accordingly and every downstream dimension follows from it.")

prices.head()

In [ ]:
# ── Quick look at the universe ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

norm_prices = prices / prices.iloc[0]
axes[0].plot(norm_prices.index, norm_prices.values, alpha=0.35, lw=0.8)
axes[0].plot(norm_prices.index, norm_prices.mean(axis=1),
             color="black", lw=2.2, label="Equal-weight mean")
axes[0].set_title("Normalized prices (base = 1.0)")
axes[0].legend()

daily_ret = np.log(prices / prices.shift(1)).dropna()
axes[1].plot(daily_ret.index, daily_ret.mean(axis=1), lw=0.8, color="#2196F3")
axes[1].set_title("Cross-sectional mean daily log return")
axes[1].axhline(0, color="black", lw=0.8)

plt.tight_layout(); plt.show()

print(f"Trading days      : {len(prices)}")
print(f"Mean daily return : {daily_ret.values.mean():.5f}")
print(f"Daily volatility  : {daily_ret.values.std():.5f}")
print(f"Annualized vol    : {daily_ret.values.std() * np.sqrt(252):.3f}")

---
## 5. Feature engineering

The **Preprocessing Module** from the architecture diagram. Per asset it builds:

| Family | Features | Count / asset |
|--------|----------|---------------|
| Log returns | `ln(P_t / P_{t-1})` | 1 |
| Realized volatility | 5 / 10 / 20-day rolling σ, annualized | 3 |
| Momentum | 5 / 10 / 20-day rate of change | 3 |
| RSI | 14-day | 1 |
| MACD | line, signal, histogram (12/26/9) | 3 |
| Bollinger | %B (20-day, 2σ) | 1 |
| Volume | volume / rolling-mean ratio, 5 / 10 / 20-day | 3 |
| | **Total** | **15** |

So the feature count is `15 × n_assets`. Rows containing NaNs from the rolling
windows are dropped, which trims roughly the first 26 observations.

In [ ]:
preprocessor = Preprocessor(
    window_size=config.data.window_size,
    normalize_method=config.data.normalize_method,
)

features_all = preprocessor.engineer_features(prices, volumes, macro_data=None)

# Re-align price/volume matrices to the surviving feature index
prices  = prices.loc[features_all.index]
volumes = volumes.loc[features_all.index]

n_features = features_all.shape[1]
print(f"Features   : {n_features}  ({n_features / config.data.n_assets:.0f} per asset)")
print(f"Timesteps  : {len(features_all)}")
print(f"Date range : {features_all.index.min().date()} → {features_all.index.max().date()}")
print(f"NaNs       : {features_all.isna().sum().sum()}")

features_all.iloc[:3, :6]

---
## 6. Chronological splits, normalization, windowing

Three points that matter for the validity of the results:

1. **Chronological, not random.** Splits are contiguous in time — no shuffling.
2. **The scaler is fit on train only.** Val and test use `transform_normalize`, so
   no distributional information leaks backwards.
3. **Prices are aligned to window *end* dates.** Window `i` spans rows
   `[i, i+30)`; the agent acts on the close of row `i+29` and earns the return
   from `i+29 → i+30`. No lookahead.

In [ ]:
n_total = len(features_all)
n_train = int(n_total * config.data.train_ratio)
n_val   = int(n_total * config.data.val_ratio)

features_train = features_all.iloc[:n_train]
features_val   = features_all.iloc[n_train:n_train + n_val]
features_test  = features_all.iloc[n_train + n_val:]

prices_train = prices.loc[features_train.index]
prices_val   = prices.loc[features_val.index]
prices_test  = prices.loc[features_test.index]

# ── Normalize: FIT ON TRAIN ONLY ───────────────────────────────────────
norm_train = preprocessor.fit_normalize(features_train)
norm_val   = preprocessor.transform_normalize(features_val)
norm_test  = preprocessor.transform_normalize(features_test)

# ── Sliding windows → x_t ──────────────────────────────────────────────
windows_train = preprocessor.create_windows(norm_train.values)
windows_val   = preprocessor.create_windows(norm_val.values)
windows_test  = preprocessor.create_windows(norm_test.values)

W = config.data.window_size
prices_train_aligned = prices_train.iloc[W - 1:].values
prices_val_aligned   = prices_val.iloc[W - 1:].values
prices_test_aligned  = prices_test.iloc[W - 1:].values

dates_train = features_train.index[W - 1:]
dates_val   = features_val.index[W - 1:]
dates_test  = features_test.index[W - 1:]

split_table = pd.DataFrame({
    "rows":    [len(features_train), len(features_val), len(features_test)],
    "windows": [len(windows_train), len(windows_val), len(windows_test)],
    "start":   [d[0].date()  for d in (dates_train, dates_val, dates_test)],
    "end":     [d[-1].date() for d in (dates_train, dates_val, dates_test)],
}, index=["train", "val", "test"])

print(split_table.to_string(), "\n")
print(f"Observation tensor x_t : {windows_train.shape[1:]}  (window_size, n_features)")

---
## 7. Market environments

`MarketEnvironment` executes trades, charges transaction costs and slippage on
turnover, and tracks portfolio value. One instance per split.

In [ ]:
def make_env(prices_arr, windows_arr):
    return MarketEnvironment(
        prices=prices_arr,
        features=windows_arr,
        initial_value=config.environment.initial_portfolio_value,
        transaction_cost_rate=config.reward.transaction_cost_rate,
        slippage=config.environment.slippage,
        allow_short=config.environment.allow_short_selling,
        max_position_size=config.environment.max_position_size,
    )

train_env = make_env(prices_train_aligned, windows_train)
val_env   = make_env(prices_val_aligned,   windows_val)
test_env  = make_env(prices_test_aligned,  windows_test)

for name, env in [("train", train_env), ("val", val_env), ("test", test_env)]:
    print(f"{name:<6} steps={env.n_steps:<5} assets={env.n_assets}")

---
## 8. Correctness patches  ⚠️

Two defects in the current source affect this pipeline. Both are patched at
runtime below — **the notebook does not modify your source files.** Set
`APPLY_CORRECTNESS_PATCHES = False` to reproduce the unpatched behaviour.

---

### Patch 1 — PPO importance ratio (affects training)

In `src/models/actor.py`, `forward()` computes `log_prob` on the **pre-softmax**
Gaussian sample, then returns the **post-softmax** action:

```python
action   = dist.rsample()                    # raw
log_prob = dist.log_prob(action).sum(-1)     # ← density of the RAW sample
action   = self._normalize_weights(action)   # ← but the SOFTMAX action is returned
```

The trainer stores the softmax action, so at update time `get_log_prob()`
evaluates the density of a *different* quantity than the stored `old_log_prob`.
The ratio `exp(new − old)` therefore compares two different random variables and
is not a valid importance weight — clipping operates on a meaningless number.

The patch computes `log_prob` on the **same** (normalized) action that gets stored,
so numerator and denominator are consistent and the ratio is exactly 1.0 when the
policy has not changed.

> **Caveat, stated plainly:** this is the *consistent* fix, not the *exact* one.
> A fully rigorous treatment needs a change-of-variables (Jacobian) correction for
> the softmax squash, or storing the raw pre-squash sample for the ratio while
> passing the normalized one to the environment. The version below is the standard
> pragmatic approximation and is what most squashed-Gaussian PPO implementations
> do. Mention the approximation in your methods section.

### Patch 2 — `info["turnover"]` always reports 0 (affects logging only)

In `src/environment/market_env.py`, `self.current_weights` is reassigned to
`new_weights` *before* the turnover figure is computed, so
`|new_weights − self.current_weights|` is identically zero. Training is
unaffected — the reward path captures `old_weights` separately in the trainer —
but any turnover number you report would be wrong.

In [ ]:
APPLY_CORRECTNESS_PATCHES = True

if APPLY_CORRECTNESS_PATCHES:

    # ── Patch 1: consistent log-prob in ActorNetwork.forward ────────────
    def _patched_actor_forward(self, h_temp, regime_probs=None, deterministic=False):
        features = self.feature_net(h_temp)
        if regime_probs is not None:
            features = features + self.regime_embed(regime_probs)

        mean = torch.tanh(self.mean_head(features))
        std  = self.log_std.exp().expand_as(mean)
        dist = torch.distributions.Normal(mean, std)

        raw_action = mean if deterministic else dist.rsample()

        # Normalize FIRST, then score the action that is actually stored/executed.
        action = self._normalize_weights(raw_action)

        log_prob = dist.log_prob(action).sum(dim=-1)
        entropy  = dist.entropy().sum(dim=-1)
        return action, log_prob, entropy

    ActorNetwork.forward = _patched_actor_forward

    # ── Patch 2: correct turnover in MarketEnvironment.step ─────────────
    _orig_step = MarketEnvironment.step

    def _patched_env_step(self, action):
        prev_weights = self.current_weights.copy()
        obs, ret, done, info = _orig_step(self, action)
        info["turnover"] = float(np.abs(self.current_weights - prev_weights).sum())
        return obs, ret, done, info

    MarketEnvironment.step = _patched_env_step

    print("✓ Patch 1 applied — PPO ratio now compares matching actions")
    print("✓ Patch 2 applied — info['turnover'] now reports real turnover")
else:
    print("⚠ Patches DISABLED — PPO importance ratio is invalid; results are not trustworthy")

---
## 9. Model construction

`PPOAgent` wires the Differentiable Regime Encoder to the Actor and Critic.
Everything below is one `nn.Module`, so a single `backward()` updates the VSN,
LSTM, MHA, GAT, fusion layer, regime classifier, actor and critic together — the
"Joint End-to-End Gradient Update" arrows in the architecture diagram.

In [ ]:
set_seed(SEED)

n_features = windows_train.shape[2]

agent = PPOAgent(
    n_features=n_features,
    n_assets=config.data.n_assets,
    n_macro=config.data.n_macro_variables,
    vsn_hidden_dim=config.regime_encoder.vsn_hidden_dim,
    lstm_hidden_dim=config.regime_encoder.lstm_hidden_dim,
    lstm_num_layers=config.regime_encoder.lstm_num_layers,
    mha_num_heads=config.regime_encoder.mha_num_heads,
    gnn_hidden_dim=config.regime_encoder.gnn_hidden_dim,
    gnn_num_heads=config.regime_encoder.gnn_num_heads,
    gnn_num_layers=config.regime_encoder.gnn_num_layers,
    n_regimes=config.regime_encoder.n_regimes,
    latent_state_dim=config.regime_encoder.latent_state_dim,
    actor_hidden_dims=config.actor.hidden_dims,
    critic_hidden_dims=config.critic.hidden_dims,
    encoder_dropout=config.regime_encoder.lstm_dropout,
    mha_dropout=config.regime_encoder.mha_dropout,
    actor_dropout=config.actor.dropout,
    critic_dropout=config.critic.dropout,
    allow_short=config.environment.allow_short_selling,
).to(DEVICE)

counts = agent.count_parameters()
print("Parameter counts")
print("─" * 40)
for k, v in counts.items():
    share = v / counts["total"] * 100
    print(f"  {k:<16} {v:>12,}   {share:5.1f}%")

### 9.1 Shape smoke test

Verifies every tensor in the forward pass before committing to a training run.

In [ ]:
agent.eval()
with torch.no_grad():
    dummy = torch.FloatTensor(windows_train[:4]).to(DEVICE)
    out = agent(dummy)

B = dummy.shape[0]
expected = {
    "action":        (B, config.data.n_assets),
    "log_prob":      (B,),
    "entropy":       (B,),
    "value":         (B,),
    "h_temp":        (B, config.regime_encoder.latent_state_dim),
    "regime_probs":  (B, config.regime_encoder.n_regimes),
    "regime_logits": (B, config.regime_encoder.n_regimes),
    "var_weights":   (B, config.data.window_size, n_features),
    "attn_weights":  (B, config.regime_encoder.mha_num_heads,
                      config.data.window_size, config.data.window_size),
}

print(f"input x_t : {tuple(dummy.shape)}\n")
all_ok = True
for key, exp in expected.items():
    got = tuple(out[key].shape)
    ok = got == exp
    all_ok &= ok
    print(f"  {'✓' if ok else '✗'} {key:<15} {str(got):<28} expected {exp}")

# Sanity: portfolio weights must form a valid simplex
w = out["action"].cpu().numpy()
print(f"\nweights sum to 1 : {np.allclose(w.sum(axis=1), 1.0)}  (row sums {w.sum(axis=1).round(6)})")
print(f"weights >= 0     : {bool((w >= 0).all())}")
print(f"regime probs sum : {out['regime_probs'].sum(dim=1).cpu().numpy().round(6)}")
assert all_ok, "Shape mismatch — stop and investigate before training."
print("\nAll shape checks passed.")

---
## 10. Training

The PPO loop per iteration:

1. **Collect rollout** — step the policy through `train_env` for `rollout_length` steps,
   recording observations, actions, log-probs, values and rewards.
2. **Reward** — `Sharpe(20d, annualized) − EVaR penalty − (linear + quadratic turnover costs)`.
3. **GAE(λ)** — `δ_t = r_t + γV(s_{t+1}) − V(s_t)`, accumulated backwards.
4. **Update** — `L = L_clip + 0.5·L_value + 0.01·L_entropy`, backpropagated jointly
   through encoder, actor and critic, with gradient-norm clipping at 0.5.

`TOTAL_TIMESTEPS` is set low by default so the notebook runs end-to-end quickly.
Raise it to 200k+ for results worth reporting.

In [ ]:
run_id  = datetime.now().strftime("%Y%m%d_%H%M%S")
log_dir = os.path.join(LOGS_DIR, f"nb_{run_id}")

trainer = PPOTrainer(
    agent=agent,
    train_env=train_env,
    val_env=val_env,
    clip_epsilon=config.ppo.clip_epsilon,
    gamma=config.ppo.gamma,
    gae_lambda=config.ppo.gae_lambda,
    entropy_coeff=config.ppo.entropy_coeff,
    value_loss_coeff=config.ppo.value_loss_coeff,
    learning_rate=config.ppo.learning_rate,
    max_grad_norm=config.ppo.max_grad_norm,
    batch_size=config.ppo.batch_size,
    n_epochs=config.ppo.n_epochs,
    rollout_length=config.ppo.rollout_length,
    total_timesteps=config.ppo.total_timesteps,
    eval_frequency=config.ppo.eval_frequency,
    save_frequency=config.ppo.save_frequency,
    early_stopping_patience=config.ppo.early_stopping_patience,
    log_dir=log_dir,
    save_dir=MODELS_DIR,
    device=DEVICE,
)

print(f"run id  : {run_id}")
print(f"logs    : {log_dir}")
print(f"models  : {MODELS_DIR}")
print(f"updates : ~{config.ppo.total_timesteps // config.ppo.rollout_length} rollouts")

In [ ]:
%%time
set_seed(SEED)
history = trainer.train()
print("\nTraining complete.")

### 10.1 Training curves

In [ ]:
def _series(hist, key):
    if isinstance(hist, dict) and key in hist:
        return np.asarray(hist[key], dtype=float)
    if isinstance(hist, list) and hist and isinstance(hist[0], dict) and key in hist[0]:
        return np.asarray([h[key] for h in hist], dtype=float)
    return None

panels = [
    ("total_loss",   "Total loss"),
    ("policy_loss",  "Policy loss (L_clip)"),
    ("value_loss",   "Value loss (MSE)"),
    ("entropy_loss", "Entropy loss"),
    ("approx_kl",    "Approx. KL"),
    ("clip_fraction","Clip fraction"),
]
available = [(k, t) for k, t in panels if _series(history, k) is not None]

if available:
    ncols = 3
    nrows = int(np.ceil(len(available) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.6 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax, (key, title) in zip(axes, available):
        y = _series(history, key)
        ax.plot(y, lw=1.4, color="#2196F3")
        if len(y) >= 10:
            k = max(3, len(y) // 12)
            ax.plot(np.convolve(y, np.ones(k) / k, mode="valid"),
                    lw=2.0, color="#F44336", label=f"MA({k})")
            ax.legend(fontsize=8)
        ax.set_title(title, fontsize=10)
        ax.set_xlabel("update")
    for ax in axes[len(available):]:
        ax.axis("off")
    plt.tight_layout(); plt.show()
else:
    print("No per-update history returned by trainer.train().")
    print("TensorBoard has the full record:  %tensorboard --logdir", log_dir)
    print("Returned keys:", list(history.keys()) if isinstance(history, dict) else type(history))

---
## 11. Evaluation & backtest

`run_episode` walks an environment deterministically (policy mean, no sampling)
and records everything needed for analysis: portfolio values, weights, regime
probabilities and the MHA attention maps.

The LSTM hidden state is carried across steps so the encoder sees a continuous
history, matching how the model would be deployed.

In [ ]:
@torch.no_grad()
def run_episode(agent, env, device=DEVICE, deterministic=True, capture_attention=True):
    '''Roll a policy through an environment; return a dict of trajectories.'''
    agent.eval()
    obs = env.reset()
    lstm_hidden = None
    done = False

    regime_probs, values, attn_maps, turnovers = [], [], [], []

    while not done:
        x = torch.FloatTensor(obs["features"]).unsqueeze(0).to(device)
        out = agent(x, lstm_hidden=lstm_hidden, deterministic=deterministic)
        lstm_hidden = out["lstm_hidden"]

        regime_probs.append(out["regime_probs"].cpu().numpy()[0])
        values.append(float(out["value"].cpu().item()))
        if capture_attention:
            # mean over heads → (window, window)
            attn_maps.append(out["attn_weights"].mean(dim=1).cpu().numpy()[0])

        action = out["action"].cpu().numpy()[0]
        next_obs, _, done, info = env.step(action)
        turnovers.append(info.get("turnover", np.nan))

        if next_obs is None:
            break
        obs = next_obs

    pv = np.asarray(env.portfolio_history, dtype=float)
    return {
        "portfolio_values": pv,
        "returns":          np.asarray(env.return_history, dtype=float),
        "weights":          np.asarray(env.weight_history, dtype=float),
        "costs":            np.asarray(env.cost_history, dtype=float),
        "regime_probs":     np.asarray(regime_probs, dtype=float),
        "values":           np.asarray(values, dtype=float),
        "turnover":         np.asarray(turnovers, dtype=float),
        "attention":        np.asarray(attn_maps, dtype=float) if attn_maps else None,
        "summary":          env.get_performance_summary(),
    }

In [ ]:
# ── Load the best checkpoint if the trainer saved one ──────────────────
best_ckpt = os.path.join(MODELS_DIR, "best_model.pt")
if os.path.exists(best_ckpt):
    trainer.load_checkpoint(best_ckpt)
    print(f"Loaded best checkpoint: {best_ckpt}")
else:
    print("No best_model.pt found — evaluating the final in-memory weights.")

# ── Run all three splits ───────────────────────────────────────────────
results = {}
for name, env in [("train", train_env), ("val", val_env), ("test", test_env)]:
    results[name] = run_episode(agent, env)
    s = results[name]["summary"]
    print(f"\n{name.upper():<6} "
          f"total_return={s['total_return']:+.4f}  "
          f"sharpe={s['sharpe_ratio']:+.3f}  "
          f"max_dd={s['max_drawdown']:.4f}  "
          f"final={s['final_portfolio_value']:,.0f} LKR")

### 11.1 Full metric suite

`PortfolioMetrics` adds Sortino, Calmar, VaR / CVaR / EVaR, win rate and profit
factor on top of the environment's own summary.

In [ ]:
metrics_calc = PortfolioMetrics(risk_free_rate=config.reward.risk_free_rate)

rows = {}
for name, res in results.items():
    rows[name] = metrics_calc.compute_all(
        returns=res["returns"],
        portfolio_values=res["portfolio_values"],
    )

metrics_df = pd.DataFrame(rows)
order = [
    "total_return", "annualized_return", "annualized_volatility",
    "sharpe_ratio", "sortino_ratio", "calmar_ratio", "max_drawdown",
    "var_95", "cvar_95", "evar_95", "win_rate", "profit_factor", "final_value",
]
metrics_df = metrics_df.reindex([r for r in order if r in metrics_df.index])

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
metrics_df

---
## 12. Benchmarks

An RL portfolio result is meaningless without a baseline. Two cheap, honest ones:

- **Equal-weight buy & hold** — allocate `1/N` on day one, never rebalance.
  No turnover, so no transaction costs.
- **Equal-weight daily rebalance** — reset to `1/N` every day, paying the same
  transaction-cost and slippage rates as the agent.

If the agent cannot beat these on the **test** split, it has not learned anything
worth reporting.

In [ ]:
def buy_and_hold(prices_arr, initial_value):
    w0 = np.ones(prices_arr.shape[1]) / prices_arr.shape[1]
    shares = initial_value * w0 / (prices_arr[0] + 1e-10)
    pv = (shares[None, :] * prices_arr).sum(axis=1)
    return pv, pv[1:] / pv[:-1] - 1.0


def equal_weight_rebalanced(prices_arr, initial_value, tc_rate, slippage):
    n = prices_arr.shape[1]
    target = np.ones(n) / n
    pv, w = initial_value, np.zeros(n)
    values, rets = [pv], []
    for t in range(len(prices_arr) - 1):
        turnover = np.abs(target - w).sum()
        cost = (tc_rate + slippage) * turnover * pv
        r = float(np.dot(target, (prices_arr[t + 1] - prices_arr[t]) / (prices_arr[t] + 1e-10)))
        pv = pv * (1 + r) - cost
        w = target
        values.append(pv); rets.append(r)
    return np.asarray(values), np.asarray(rets)


init_val = config.environment.initial_portfolio_value
bh_values, bh_returns = buy_and_hold(prices_test_aligned, init_val)
ew_values, ew_returns = equal_weight_rebalanced(
    prices_test_aligned, init_val,
    config.reward.transaction_cost_rate, config.environment.slippage,
)

comparison = pd.DataFrame({
    "PPO (agent)":        metrics_calc.compute_all(results["test"]["returns"],
                                                   results["test"]["portfolio_values"]),
    "Equal-wt buy&hold":  metrics_calc.compute_all(bh_returns, bh_values),
    "Equal-wt rebalanced":metrics_calc.compute_all(ew_returns, ew_values),
}).reindex([r for r in order if r in metrics_df.index])

print("TEST SPLIT — agent vs. benchmarks\n")
comparison

---
## 13. Visualization

In [ ]:
res = results["test"]
pv  = res["portfolio_values"]

fig, axes = plt.subplots(2, 1, figsize=(14, 8),
                         gridspec_kw={"height_ratios": [3, 1]}, sharex=True)

axes[0].plot(pv, lw=2.0, color="#2196F3", label="PPO agent")
axes[0].plot(bh_values, lw=1.5, color="#FF9800", alpha=0.85, label="Equal-wt buy & hold")
axes[0].plot(ew_values, lw=1.5, color="#4CAF50", alpha=0.85, label="Equal-wt rebalanced")
axes[0].axhline(init_val, color="black", ls="--", lw=0.9, alpha=0.6)
axes[0].set_ylabel("Portfolio value (LKR)")
axes[0].set_title("Test-split equity curve", fontweight="bold")
axes[0].legend()

peak = np.maximum.accumulate(pv)
dd = (peak - pv) / (peak + 1e-10)
axes[1].fill_between(range(len(dd)), dd, alpha=0.45, color="#F44336")
axes[1].set_ylabel("Drawdown"); axes[1].set_xlabel("Trading day")
axes[1].invert_yaxis()

plt.tight_layout(); plt.show()

In [ ]:
# ── Learned latent regimes ─────────────────────────────────────────────
rp = res["regime_probs"]
regime_names = ["Bull", "Bear", "Sideways"][:rp.shape[1]]
colors = ["#4CAF50", "#F44336", "#9E9E9E"][:rp.shape[1]]

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True,
                         gridspec_kw={"height_ratios": [2, 2, 1.4]})

axes[0].stackplot(range(len(rp)), rp.T, labels=regime_names, colors=colors, alpha=0.8)
axes[0].set_ylabel("P(regime)"); axes[0].set_ylim(0, 1)
axes[0].set_title("Latent regime probabilities (learned end-to-end, no regime labels)",
                  fontweight="bold")
axes[0].legend(loc="upper right", ncol=len(regime_names))

axes[1].plot(pv[:len(rp)], lw=1.8, color="#2196F3")
dominant = rp.argmax(axis=1)
for k, c in enumerate(colors):
    axes[1].fill_between(range(len(rp)), pv[:len(rp)].min(), pv[:len(rp)].max(),
                         where=(dominant == k), color=c, alpha=0.14)
axes[1].set_ylabel("Portfolio value")
axes[1].set_title("Equity curve shaded by dominant regime")

ent = -(rp * np.log(rp + 1e-10)).sum(axis=1)
axes[2].plot(ent, lw=1.3, color="#673AB7")
axes[2].axhline(np.log(rp.shape[1]), color="black", ls="--", lw=0.9,
                label=f"max = ln({rp.shape[1]}) = {np.log(rp.shape[1]):.3f}")
axes[2].set_ylabel("Regime entropy"); axes[2].set_xlabel("Trading day")
axes[2].legend(fontsize=8)

plt.tight_layout(); plt.show()

print("Mean regime probabilities:")
for nme, p in zip(regime_names, rp.mean(axis=0)):
    print(f"  {nme:<10} {p:.4f}")
print(f"\nMean entropy {ent.mean():.4f} / max {np.log(rp.shape[1]):.4f}"
      f"   → {'confident, well-separated regimes' if ent.mean() < 0.8 * np.log(rp.shape[1]) else 'near-uniform: regimes are NOT separating'}")

In [ ]:
# ── Portfolio allocation over time ─────────────────────────────────────
W_hist = res["weights"]
asset_names = config.data.asset_names

fig, axes = plt.subplots(1, 2, figsize=(15, 5),
                         gridspec_kw={"width_ratios": [2, 1]})

im = axes[0].imshow(W_hist.T, aspect="auto", cmap="viridis",
                    interpolation="nearest", origin="lower")
axes[0].set_xlabel("Trading day"); axes[0].set_ylabel("Asset")
axes[0].set_title("Portfolio weights over time", fontweight="bold")
plt.colorbar(im, ax=axes[0], label="weight")

mean_w = W_hist[1:].mean(axis=0)
top = np.argsort(mean_w)[::-1][:15]
axes[1].barh([asset_names[i] for i in top][::-1], mean_w[top][::-1], color="#2196F3")
axes[1].axvline(1 / len(asset_names), color="#F44336", ls="--", lw=1.2,
                label=f"equal weight = {1/len(asset_names):.4f}")
axes[1].set_xlabel("Mean weight"); axes[1].set_title("Top 15 holdings")
axes[1].legend(fontsize=8)

plt.tight_layout(); plt.show()

hhi = (W_hist[1:] ** 2).sum(axis=1)
print(f"Mean turnover / day : {np.nanmean(res['turnover']):.4f}")
print(f"Mean HHI            : {hhi.mean():.4f}  (equal weight = {1/len(asset_names):.4f}, fully concentrated = 1.0)")
print(f"Effective # assets  : {1/hhi.mean():.1f} of {len(asset_names)}")
print(f"Total costs paid    : {res['costs'].sum():,.0f} LKR "
      f"({res['costs'].sum()/init_val*100:.2f}% of initial capital)")

In [ ]:
# ── Temporal attention (MHA) ───────────────────────────────────────────
if res["attention"] is not None:
    attn = res["attention"]                     # (T, window, window)
    mean_attn = attn.mean(axis=0)               # averaged over the episode

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    im = axes[0].imshow(mean_attn, cmap="magma", aspect="auto", origin="lower")
    axes[0].set_xlabel("Key timestep (lag)"); axes[0].set_ylabel("Query timestep")
    axes[0].set_title("Mean MHA attention map (heads averaged)", fontweight="bold")
    plt.colorbar(im, ax=axes[0])

    # The decision actually uses the LAST query row (see regime_encoder.py:143)
    last_row = attn[:, -1, :].mean(axis=0)
    axes[1].bar(range(len(last_row)), last_row, color="#673AB7")
    axes[1].set_xlabel("Timestep within 30-day window (0 = oldest)")
    axes[1].set_ylabel("Attention weight")
    axes[1].set_title("Attention from the decision timestep", fontweight="bold")
    axes[1].axhline(1 / len(last_row), color="#F44336", ls="--", lw=1.2,
                    label=f"uniform = {1/len(last_row):.4f}")
    axes[1].legend(fontsize=8)

    plt.tight_layout(); plt.show()

    top_lags = np.argsort(last_row)[::-1][:5]
    print("Most-attended lags (0 = oldest day in window, 29 = most recent):")
    for i in top_lags:
        print(f"  t-{len(last_row)-1-i:<3} (index {i:2d})  weight {last_row[i]:.4f}")
else:
    print("Attention was not captured.")

---
## 14. Saving artifacts

Everything needed to reproduce or write up this run is written to `results/`.

In [ ]:
out_dir = Path(RESULTS_DIR) / f"nb_run_{run_id}"
out_dir.mkdir(parents=True, exist_ok=True)

# 1. Metrics
metrics_df.to_csv(out_dir / "metrics_by_split.csv")
comparison.to_csv(out_dir / "test_vs_benchmarks.csv")

# 2. Per-step trajectory (test split)
traj = pd.DataFrame({
    "date":             dates_test[:len(res["portfolio_values"])],
    "portfolio_value":  res["portfolio_values"],
})
traj["portfolio_return"] = np.r_[0.0, res["returns"]][:len(traj)]
traj["cost"]             = np.r_[0.0, res["costs"]][:len(traj)]
for k, nme in enumerate(regime_names):
    traj[f"p_{nme.lower()}"] = np.r_[np.nan, rp[:, k]][:len(traj)]
for i, a in enumerate(asset_names):
    traj[f"w_{a}"] = res["weights"][:len(traj), i]
traj.to_csv(out_dir / "test_trajectory.csv", index=False)

# 3. Run manifest
manifest = {
    "run_id": run_id,
    "timestamp": datetime.now().isoformat(),
    "seed": SEED,
    "device": DEVICE,
    "patches_applied": APPLY_CORRECTNESS_PATCHES,
    "n_assets": config.data.n_assets,
    "n_features": int(n_features),
    "window_size": config.data.window_size,
    "total_timesteps": config.ppo.total_timesteps,
    "parameters": counts,
    "splits": {k: int(v) for k, v in
               zip(["train", "val", "test"],
                   [len(windows_train), len(windows_val), len(windows_test)])},
    "test_summary": {k: float(v) for k, v in results["test"]["summary"].items()},
}
(out_dir / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

# 4. Model checkpoint
torch.save({
    "model_state_dict": agent.state_dict(),
    "config": manifest,
}, out_dir / "agent_final.pt")

print(f"Saved to {out_dir}\n")
for p in sorted(out_dir.iterdir()):
    print(f"  {p.name:<28} {p.stat().st_size/1024:>9,.1f} KB")

---
## Notes & next steps

**Before reporting any numbers:**

1. **Raise `TOTAL_TIMESTEPS`.** The 20k default exists so the notebook runs
   end-to-end in minutes. It is far too short to converge.
2. **Run multiple seeds.** RL results on a single seed are noise. Report
   mean ± std over at least 5 seeds.
3. **Check the regime entropy plot.** If entropy sits near `ln(3) ≈ 1.099` the
   regime classifier is outputting near-uniform probabilities and is not
   separating anything — the "regime detection" claim would not be supported.
4. **Beat the benchmarks in §12.** On the *test* split, not train.

**Known limitations in the current code**, in the order they matter:

| # | Issue | Where | Status |
|---|-------|-------|--------|
| 1 | PPO ratio compared mismatched actions | `src/models/actor.py` | Patched in §8 (approximate) |
| 2 | `info["turnover"]` always 0 | `src/environment/market_env.py` | Patched in §8 |
| 3 | `DifferentiableReward` defined but unused | `src/environment/reward.py` | Harmless — but don't claim gradients flow *through* the reward; in PPO they reach the encoder via the advantage only |
| 4 | Only the last MHA query row reaches `h_temp` | `src/models/regime_encoder.py` | Works correctly; attention-pooling with a single learnable query would be ~30× cheaper in that block |

**To make patch 1 exact** rather than approximate, change `ActorNetwork.forward`
to return the raw pre-softmax sample alongside the normalized action, store the
raw one for the PPO ratio, and pass the normalized one to the environment. That
removes the missing-Jacobian approximation entirely.